# 🎓 SmolSocrates-360M: Fine-Tuning a Socratic Coding Tutor

Fine-tune **SmolLM2-360M** with **LoRA + Unsloth** to create a tiny model that teaches coding through Socratic questioning — never giving direct answers, always guiding through progressive hints.

**What you'll learn:**
- How to fine-tune a 360M parameter model on a narrow pedagogical task
- LoRA configuration for efficient training
- Before vs. after evaluation with a custom Socratic rubric
- Push your model to HuggingFace Hub

⏱️ **Runtime**: ~15 minutes on a free T4 GPU

---

## 1. Setup & Install Dependencies

In [1]:

# Install Unsloth (optimized for Colab T4)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets groq python-dotenv

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-5f8eh6by/unsloth_16524740193d488db17d67b4a6eb93dc
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-5f8eh6by/unsloth_16524740193d488db17d67b4a6eb93dc
  Resolved https://github.com/unslothai/unsloth.git to commit 1798ebe3f30281978abdc7e409bd91a11a1faa8a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformer

In [2]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


## 2. Load Base Model

In [3]:
from unsloth import FastLanguageModel
import torch

# Configuration
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # Use 4-bit quantization to save memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"✅ Loaded {MODEL_NAME}")
print(f"   Parameters: {model.num_parameters():,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

HuggingFaceTB/SmolLM2-360M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.
✅ Loaded HuggingFaceTB/SmolLM2-360M-Instruct
   Parameters: 361,821,120


## 3. Apply LoRA Adapters

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

# Print trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n🔧 LoRA applied!")
print(f"   Trainable: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Total:     {total:,}")

Unsloth 2026.5.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.



🔧 LoRA applied!
   Trainable: 8,683,520 (4.07%)
   Total:     213,218,240


## 4. Load & Prepare Dataset

In [5]:
from datasets import load_dataset
import random

# Our Socratic tutor system prompt
SYSTEM_PROMPT = (
    "You are a Socratic coding tutor. Your role is to help students learn programming "
    "by asking guiding questions — never by giving direct answers or complete solutions.\n\n"
    "Rules:\n"
    "1. NEVER provide complete code solutions or direct fixes.\n"
    "2. Ask targeted questions that lead the student to discover the answer themselves.\n"
    "3. Break complex problems into smaller, manageable steps.\n"
    "4. If a student is stuck, offer a progressive hint — start broad, get specific.\n"
    "5. Celebrate progress and encourage self-discovery.\n"
    "6. Match the student's technical level in your language.\n"
    "7. When debugging, ask the student to trace through their code mentally."
)

# Load the PACT dataset
ds = load_dataset("AndreiSobo/PACT-Socratic-Coding-Tutor", split="train")
print(f"📥 Loaded {len(ds)} examples")

# Process: standardize system prompts
def process_example(example):
    messages = example["messages"]
    processed = []
    has_system = False

    for msg in messages:
        if msg["role"] == "system":
            processed.append({"role": "system", "content": SYSTEM_PROMPT})
            has_system = True
        else:
            processed.append({"role": msg["role"], "content": msg["content"]})

    if not has_system:
        processed.insert(0, {"role": "system", "content": SYSTEM_PROMPT})

    return {"messages": processed}

ds = ds.map(process_example)

# Split: 197 train / 30 eval
ds = ds.shuffle(seed=42)
split = ds.train_test_split(test_size=30, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"   Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Preview a sample
sample = train_dataset[0]["messages"]
for msg in sample:
    role = msg["role"].upper()
    content = msg["content"][:100] + "..." if len(msg["content"]) > 100 else msg["content"]
    print(f"   [{role}] {content}")

README.md:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/99.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/227 [00:00<?, ? examples/s]

📥 Loaded 227 examples


Map:   0%|          | 0/227 [00:00<?, ? examples/s]

   Train: 197 | Eval: 30
   [SYSTEM] You are a Socratic coding tutor. Your role is to help students learn programming by asking guiding q...
   [USER] Problem: Permutation Sequence

The set [1, 2, 3, ..., n] contains a total of n! unique permutations....
   [ASSISTANT] When you're choosing a digit for a certain position, think about how many positions remain after you...


## 5. Baseline Evaluation (Before Fine-Tuning)

In [6]:
import re

def quick_socratic_score(response: str) -> dict:
    """Quick rule-based scoring for Socratic quality."""

    # 1. Code blocks present? (lower is better for Socratic)
    code_blocks = len(re.findall(r"```[\s\S]*?```", response))
    no_code_score = 3 if code_blocks == 0 else (1 if code_blocks == 1 else 0)

    # 2. Questions asked?
    questions = [s for s in re.split(r"[.!\n]", response) if "?" in s and len(s.strip()) > 10]
    question_score = min(3, len(questions))

    # 3. Encouraging tone?
    encouraging = ["great", "good", "think about", "consider", "let's", "try", "what if"]
    enc_count = sum(1 for p in encouraging if p in response.lower())
    enc_score = min(3, enc_count)

    total = no_code_score + question_score + enc_score
    return {
        "no_code": no_code_score,
        "questions": question_score,
        "encouragement": enc_score,
        "total": total,
        "max": 9,
        "code_blocks": code_blocks,
        "num_questions": len(questions),
    }


# Test prompts for before/after comparison
TEST_PROMPTS = [
    "How do I reverse a string in Python?",
    "My for loop never stops running. Here's my code:\n```python\ni = 0\nwhile i < 10:\n    print(i)\n```\nWhat's wrong?",
    "What's the difference between a list and a tuple in Python?",
    "I'm getting an IndexError in my code. How do I fix it?",
    "Can you explain recursion to me? I don't get it.",
]


# Run baseline evaluation
FastLanguageModel.for_inference(model)

print("=" * 60)
print("📊 BASELINE EVALUATION (Before Fine-Tuning)")
print("=" * 60)

baseline_results = []

for i, prompt in enumerate(TEST_PROMPTS):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    score = quick_socratic_score(response)
    baseline_results.append({"prompt": prompt, "response": response, "score": score})

    print(f"\n--- Test {i+1} (Score: {score['total']}/{score['max']}) ---")
    print(f"Q: {prompt[:80]}")
    print(f"A: {response[:200]}..." if len(response) > 200 else f"A: {response}")

baseline_avg = sum(r["score"]["total"] for r in baseline_results) / len(baseline_results)
print(f"\n📊 Baseline Average: {baseline_avg:.1f}/9")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


📊 BASELINE EVALUATION (Before Fine-Tuning)


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/


--- Test 1 (Score: 0/9) ---
Q: How do I reverse a string in Python?
A: You can reverse a string in Python using the following methods:

1. Using the `reversed` function:

```python
def reverse_string(input_str):
    return "".join(reversed(input_str))

input_str = "Hello...


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Test 2 (Score: 1/9) ---
Q: My for loop never stops running. Here's my code:
```python
i = 0
while i < 10:
 
A: The issue is with the condition of your while loop. You're iterating over the range of numbers from 0 to 9, which is a single-digit range. This is not what you want, as it's not a valid range for a fo...


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Test 3 (Score: 3/9) ---
Q: What's the difference between a list and a tuple in Python?
A: In Python, the difference between a list and a tuple is primarily one of data structure and order.

A list is a collection of items that can be of any data type, including strings, integers, floats, a...


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Test 4 (Score: 6/9) ---
Q: I'm getting an IndexError in my code. How do I fix it?
A: The most likely cause for an IndexError is that you are trying to access or modify a variable that does not exist in your code. Check for any typos or missing brackets, and make sure you are using the...

--- Test 5 (Score: 1/9) ---
Q: Can you explain recursion to me? I don't get it.
A: Recursion is a programming technique where a function calls itself repeatedly until it reaches a base case that stops the recursion. It's a way to solve problems by breaking them down into smaller sub...

📊 Baseline Average: 2.2/9


## 6. Fine-Tune with SFTTrainer

In [19]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

FastLanguageModel.for_training(model)

# Pre-process: convert messages to text BEFORE passing to trainer
def convert_to_text(example):
    msgs = example["messages"]
    if isinstance(msgs, dict):
        converted = [{"role": r, "content": c} for r, c in zip(msgs["role"], msgs["content"])]
    else:
        converted = msgs
    text = tokenizer.apply_chat_template(converted, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_text = train_dataset.map(convert_to_text)
eval_text = eval_dataset.map(convert_to_text)

# Verify it worked
print("Sample text:")
print(train_text[0]["text"][:300])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,       # ← smaller batch
        gradient_accumulation_steps=2,       # ← less accumulation
        num_train_epochs=10,                 # ← 10 epochs instead of 3
        warmup_steps=5,
        learning_rate=5e-4,                  # ← higher LR for small model
        lr_scheduler_type="cosine",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        output_dir="outputs",
        save_strategy="no",                  # ← skip saving to speed up
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=42,
        report_to="none",
    ),
)


print("🚀 Starting training...")
stats = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Total steps: {stats.global_step}")
print(f"   Training loss: {stats.training_loss:.4f}")
print(f"   Runtime: {stats.metrics['train_runtime']:.0f}s")


Map:   0%|          | 0/197 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Sample text:
<|im_start|>system
You are a Socratic coding tutor. Your role is to help students learn programming by asking guiding questions — never by giving direct answers or complete solutions.

Rules:
1. NEVER provide complete code solutions or direct fixes.
2. Ask targeted questions that lead the student to
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/197 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 197 | Num Epochs = 10 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 8,683,520 of 370,504,640 (2.34% trained)


Step,Training Loss
10,0.706127
20,0.591169
30,0.581859
40,0.463724
50,0.449785
60,0.332430
70,0.385256
80,0.326775
90,0.361091
100,0.328926



✅ Training complete!
   Total steps: 500
   Training loss: 0.1820
   Runtime: 585s


## 7. Post-Training Evaluation

In [20]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

print("=" * 60)
print("📊 FINE-TUNED EVALUATION (After Training)")
print("=" * 60)

finetuned_results = []

for i, prompt in enumerate(TEST_PROMPTS):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    score = quick_socratic_score(response)
    finetuned_results.append({"prompt": prompt, "response": response, "score": score})

    print(f"\n--- Test {i+1} (Score: {score['total']}/{score['max']}) ---")
    print(f"Q: {prompt[:80]}")
    print(f"A: {response[:200]}..." if len(response) > 200 else f"A: {response}")

finetuned_avg = sum(r["score"]["total"] for r in finetuned_results) / len(finetuned_results)
print(f"\n📊 Fine-tuned Average: {finetuned_avg:.1f}/9")

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📊 FINE-TUNED EVALUATION (After Training)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Test 1 (Score: 1/9) ---
Q: How do I reverse a string in Python?
A: You can use the Reversal Approach to reverse a string in Python:

```python
# Your code goes here
```

This solution works by reversing the order of characters in the string by reversing the list of c...


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Test 2 (Score: 6/9) ---
Q: My for loop never stops running. Here's my code:
```python
i = 0
while i < 10:
 
A: Your loop condition is checking whether 0 is less than 10, which is technically true because 0 is 10 minus one. However, your loop continues indefinitely because you're never out of the loop.

Think a...


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Test 3 (Score: 2/9) ---
Q: What's the difference between a list and a tuple in Python?
A: In Python, the key difference between a list and a tuple is how they are ordered and how elements can be accessed:

- Lists are ordered and can be accessed by their index.
- Tuples are ordered the sam...


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Test 4 (Score: 6/9) ---
Q: I'm getting an IndexError in my code. How do I fix it?
A: What does your code do? What does the error message say? Remember that when you get an error, think about what might be wrong and try that at the beginning of your reasoning.

--- Test 5 (Score: 9/9) ---
Q: Can you explain recursion to me? I don't get it.
A: Recursion can be a tricky concept, but let's break it down step by step.

Imagine you have a function that takes a number as input. You want to solve your problem by asking yourself the following ques...

📊 Fine-tuned Average: 4.8/9


## 8. Before vs After Comparison

In [22]:
print("\n" + "=" * 70)
print("📊 BEFORE vs AFTER COMPARISON")
print("=" * 70)
print(f"\n{'Metric':<25s} {'Baseline':>10s} {'Fine-tuned':>12s} {'Delta':>8s}")
print(f"{'-'*25} {'-'*10} {'-'*12} {'-'*8}")

# Overall
delta = finetuned_avg - baseline_avg
sign = "+" if delta > 0 else ""
print(f"{'Overall (avg)':<25s} {baseline_avg:>10.1f} {finetuned_avg:>12.1f} {sign}{delta:>7.1f}")

# Per dimension
for dim in ["no_code", "questions", "encouragement"]:
    b_avg = sum(r["score"][dim] for r in baseline_results) / len(baseline_results)
    f_avg = sum(r["score"][dim] for r in finetuned_results) / len(finetuned_results)
    delta = f_avg - b_avg
    sign = "+" if delta > 0 else ""
    print(f"{dim:<25s} {b_avg:>10.1f} {f_avg:>12.1f} {sign}{delta:>7.1f}")

# No-code rate
b_ncr = sum(1 for r in baseline_results if r["score"]["code_blocks"] == 0) / len(baseline_results) * 100
f_ncr = sum(1 for r in finetuned_results if r["score"]["code_blocks"] == 0) / len(finetuned_results) * 100
delta = f_ncr - b_ncr
sign = "+" if delta > 0 else ""
print(f"{'No-Code Rate (%)':<25s} {b_ncr:>10.0f} {f_ncr:>12.0f} {sign}{delta:>7.0f}")

print("\n" + "=" * 70)

# Side-by-side examples
print("\n📝 SIDE-BY-SIDE EXAMPLES:")
for i in range(min(3, len(TEST_PROMPTS))):
    print(f"\n{'─'*60}")
    print(f"Prompt: {TEST_PROMPTS[i][:80]}")
    print(f"\n🔴 BASELINE ({baseline_results[i]['score']['total']}/9):")
    print(f"{baseline_results[i]['response'][:200]}")
    print(f"\n🟢 FINE-TUNED ({finetuned_results[i]['score']['total']}/9):")
    print(f"{finetuned_results[i]['response'][:200]}")


📊 BEFORE vs AFTER COMPARISON

Metric                      Baseline   Fine-tuned    Delta
------------------------- ---------- ------------ --------
Overall (avg)                    2.2          4.8 +    2.6
no_code                          1.6          2.2 +    0.6
questions                        0.4          1.0 +    0.6
encouragement                    0.2          1.6 +    1.4
No-Code Rate (%)                  40           60 +     20


📝 SIDE-BY-SIDE EXAMPLES:

────────────────────────────────────────────────────────────
Prompt: How do I reverse a string in Python?

🔴 BASELINE (0/9):
You can reverse a string in Python using the following methods:

1. Using the `reversed` function:

```python
def reverse_string(input_str):
    return "".join(reversed(input_str))

input_str = "Hello

🟢 FINE-TUNED (1/9):
You can use the Reversal Approach to reverse a string in Python:

```python
# Your code goes here
```

This solution works by reversing the order of characters in the string by reve

## 9. Push to HuggingFace Hub

In [ ]:
# 🔑 Set your HuggingFace token
# Get one from: https://huggingface.co/settings/tokens
from huggingface_hub import login

# Option 1: Paste token directly (delete after pushing!)
# login(token="hf_your_token_here")

# Option 2: Use Colab secrets (recommended)
# from google.colab import userdata
# login(token=userdata.get('HF_TOKEN'))

login()  # Interactive login

In [ ]:
# ✏️ CHANGE THIS to your HuggingFace username
HF_USERNAME = "your-username"  # <-- EDIT THIS
MODEL_REPO = f"{HF_USERNAME}/SmolSocrates-360M"

# Push LoRA adapters (small, ~20MB)
print(f"📤 Pushing LoRA adapters to {MODEL_REPO}...")
model.push_to_hub(MODEL_REPO)
tokenizer.push_to_hub(MODEL_REPO)
print(f"✅ Model pushed to: https://huggingface.co/{MODEL_REPO}")

# Optionally push merged model (full, ~700MB)
# model.push_to_hub_merged(MODEL_REPO + "-merged", tokenizer)

## 10. Save Results & Export

In [ ]:
import json

# Save evaluation results
results = {
    "model_name": MODEL_REPO,
    "base_model": MODEL_NAME,
    "lora_config": {
        "r": 16,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    },
    "training": {
        "dataset": "AndreiSobo/PACT-Socratic-Coding-Tutor",
        "train_examples": len(train_dataset),
        "eval_examples": len(eval_dataset),
        "epochs": 3,
        "learning_rate": 2e-4,
    },
    "baseline_avg_score": baseline_avg,
    "finetuned_avg_score": finetuned_avg,
    "improvement": finetuned_avg - baseline_avg,
    "baseline_results": baseline_results,
    "finetuned_results": finetuned_results,
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("💾 Results saved to evaluation_results.json")
print(f"\n🎯 Final improvement: {baseline_avg:.1f} → {finetuned_avg:.1f} ({finetuned_avg - baseline_avg:+.1f})")

In [ ]:
# Download results file
from google.colab import files
files.download("evaluation_results.json")

## 🎉 Done!

**What you've accomplished:**
1. ✅ Fine-tuned SmolLM2-360M on Socratic coding tutoring
2. ✅ Evaluated before vs. after with a custom rubric
3. ✅ Pushed the model to HuggingFace Hub

**Next steps:**
- Run the full evaluation harness locally: `python eval/evaluate.py eval --model your-username/SmolSocrates-360M --output eval/results/finetuned_results.json`
- Try the interactive demo: `python src/inference.py --model your-username/SmolSocrates-360M`
- Write your blog post (draft in `blog/post.md`)